# Example: Compound

In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(project_root, "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("PYTHONPATH:", sys.path[0])

PYTHONPATH: /opt/anaconda3/envs/pybiodatafuse/lib/python310.zip


In [2]:
# Import modules
import pandas as pd

import pyBiodatafuse.constants as Cons
from pyBiodatafuse import id_mapper
from pyBiodatafuse.annotators import wikipathways
from pyBiodatafuse.graph import generator

In [3]:
bridgedb_dataframe = pd.DataFrame(
    {
        "identifier": ["ALG14", "ALG2", "CHRNA1"],
        "identifier.source": ["HGNC", "HGNC", "HGNC"],
        "target": ["199857", "85365", "1134"],
        "target.source": ["NCBI Gene", "NCBI Gene", "NCBI Gene"],
    }
)

obtained_data, metadata = wikipathways.get_gene_wikipathways(bridgedb_dataframe)

Querying WikiPathways: 100%|██████████| 1/1 [00:16<00:00, 16.25s/it]


In [4]:
obtained_data

,identifier,identifier.source,target,target.source,WikiPathways
0,ALG14,HGNC,199857,NCBI Gene,"[{'pathway_id': 'WikiPathways:5153', 'pathway_..."
1,ALG2,HGNC,85365,NCBI Gene,"[{'pathway_id': 'WikiPathways:5153', 'pathway_..."
2,CHRNA1,HGNC,1134,NCBI Gene,"[{'pathway_id': nan, 'pathway_label': nan, 'pa..."


In [5]:
obtained_data["WikiPathways"].values.tolist()

[[{'pathway_id': 'WikiPathways:5153',
   'pathway_label': 'N-glycan biosynthesis',
   'pathway_gene_count': 57.0}],
 [{'pathway_id': 'WikiPathways:5153',
   'pathway_label': 'N-glycan biosynthesis',
   'pathway_gene_count': 57.0}],
 [{'pathway_id': nan, 'pathway_label': nan, 'pathway_gene_count': nan}]]

In [6]:
obtained_data2, metadata2 = wikipathways.get_gene_wikipathways(
    bridgedb_dataframe, query_interactions=True
)

Querying WikiPathways: 100%|██████████| 1/1 [00:00<00:00,  4.89it/s]


In [8]:
obtained_data2

,identifier,identifier.source,target,target.source,WikiPathways_molecular
0,ALG14,HGNC,199857,NCBI Gene,None
1,ALG2,HGNC,85365,NCBI Gene,None
2,CHRNA1,HGNC,1134,NCBI Gene,None


In [12]:
genes_of_interest = """7350
6198
1499
6528
6714
10000
10891
6194
7068
4193
3709
"""


gene_list = genes_of_interest.split("\n")
data_input = pd.DataFrame(gene_list, columns=["identifier"])
k, bridgedb_metadata = id_mapper.bridgedb_xref(
    identifiers=data_input,
    input_species="Human",
    input_datasource="NCBI Gene",
    output_datasource="All",
)

In [15]:
k[k["target.source"] == "NCBI Gene"]

,identifier,identifier.source,target,target.source
22,7350,Entrez Gene,7350,NCBI Gene
266,6198,Entrez Gene,6198,NCBI Gene
615,1499,Entrez Gene,1499,NCBI Gene
872,6528,Entrez Gene,6528,NCBI Gene
1279,6714,Entrez Gene,6714,NCBI Gene
1306,10000,Entrez Gene,10000,NCBI Gene
1490,10891,Entrez Gene,10891,NCBI Gene
1774,6194,Entrez Gene,6194,NCBI Gene
2065,7068,Entrez Gene,7068,NCBI Gene
2182,4193,Entrez Gene,4193,NCBI Gene


In [7]:
from pyBiodatafuse.utils import combine_sources

k = combine_sources(
    bridgedb_dataframe,
    [obtained_data],
)

In [ ]:
pygraph = generator.build_networkx_graph(k)

In [ ]:
from collections import defaultdict

ntypes = defaultdict(int)
for node, attrs in pygraph.nodes(data=True):
    ntypes[attrs["label"]] += 1
ntypes

In [ ]:
for node, d in pygraph.nodes(data=True):
    print(node, d)

In [ ]:
for node, target, data in pygraph.edges(data=True):
    print(node, target, data)

In [ ]:
from pyBiodatafuse.graph import cytoscape

cytoscape.load_graph(pygraph, network_name="Test network")

In [ ]:
 stringId_A            stringId_B preferredName_A preferredName_B  \
0  9606.ENSP00000261007  9606.ENSP00000359224          CHRNA1           ALG14   
1  9606.ENSP00000359224  9606.ENSP00000417764           ALG14            ALG2   

  ncbiTaxonId  score  nscore  fscore  pscore  ascore  escore  dscore  tscore  
0        9606  0.543       0       0       0   0.000       0   0.000   0.543  
1        9606  0.633       0       0       0   0.067       0   0.119   0.589  

# convert above text to pandas dataframe

